# 01 — GitHub verisini okuma, tuz temizliği, duplicate kontrolü ve Lipinski store

Bu notebook, regression veri dosyasını **GitHub deposundan otomatik olarak bulur**, kimyasal temizleme işlemlerini uygular ve bütün çıktıları Google Drive'a kaydeder.

## Bu sürümde düzeltilen hata

Önceki sürüm yalnızca aşağıdaki sabit yolu deniyordu:

```text
data/CHEMBL206_IC50_single_protein_format_CLEAN_parent_dedup.csv
```

Dosya GitHub deposunda bu konumda bulunmadığında `404 Not Found` hatası oluşuyordu.

Yeni sürüm:

1. GitHub deposunun dosya ağacını API ile kontrol eder.
2. Hedefe özel CSV'yi hem `data/` klasöründe hem repo kökünde arar.
3. Repo kökündeki `training_data.csv` dosyasını da aday olarak değerlendirir.
4. İlk bulunan ve gerekli sütunları içeren CSV'yi kullanır.
5. İlk 404 hatasında durmaz; diğer adaylara geçer.

## Workshop veri akışı

```text
GitHub CSV
   ↓
Sütun ve veri kalite kontrolü
   ↓
SMILES doğrulama
   ↓
Tuz / karşı iyon temizliği
   ↓
Canonical parent SMILES
   ↓
Duplicate birleştirme
   ↓
Lipinski Rule of Five
   ↓
Google Drive çıktıları
```

Google Drive çıktı klasörü:

```text
MyDrive/MOL_FET_regression_workshop/
```

> Girdi GitHub'dan okunur. Çıktılar Google Drive'a yazılır.

## Hücre 1 — Gerekli paketlerin kurulması

In [ ]:
import importlib.util
# Python ortamında bir paketin kurulu olup olmadığını kontrol etmek için kullanılır.

import subprocess
# Eksik paketleri aktif Python ortamına kurmak için kullanılır.

import sys
# Colab oturumunda kullanılan aktif Python yorumlayıcısının yolunu almak için kullanılır.


REQUIRED_PACKAGES = [
    ("numpy", "numpy"),
    # Sayısal hesaplamalar için kullanılır.

    ("pandas", "pandas"),
    # CSV okuma, temizleme, gruplama ve çıktı kaydı için kullanılır.

    ("requests", "requests"),
    # GitHub API ve GitHub RAW bağlantılarına erişmek için kullanılır.

    ("tqdm", "tqdm"),
    # Uzun SMILES işlemlerinde ilerleme çubuğu göstermek için kullanılır.

    ("rdkit", "rdkit"),
    # Moleküler yapı okuma, tuz temizliği ve Lipinski hesaplamaları için kullanılır.
]
# Kurulması gereken paketlerin import adı ve pip adı birlikte tanımlanır.


for import_name, pip_name in REQUIRED_PACKAGES:
    # Paketler sırayla kontrol edilir.

    if importlib.util.find_spec(import_name) is None:
        # Paket aktif ortamda bulunmuyorsa kurulum yapılır.

        print(f"Kuruluyor: {pip_name}")
        # Kurulacak paketin adı kullanıcıya gösterilir.

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                pip_name,
            ]
        )
        # Paket, notebookun kullandığı aktif Python ortamına kurulur.

    else:
        # Paket zaten kuruluysa tekrar kurulum yapılmaz.

        print(f"Zaten kurulu: {import_name}")
        # Mevcut paket bilgisi ekrana yazdırılır.


print("✅ CHECKPOINT 01.1: Paket kontrolü tamamlandı.")
# Paket hücresinin hatasız tamamlandığı bildirilir.

## Hücre 2 — Kütüphanelerin içe aktarılması

In [ ]:
from pathlib import Path
# Google Drive çıktı yollarını güvenli biçimde yönetmek için kullanılır.

from io import BytesIO
# GitHub'dan indirilen CSV içeriğini bellekte pandas'a aktarmak için kullanılır.

import warnings
# Gereksiz uyarıları kontrol etmek için kullanılır.

warnings.filterwarnings("ignore")
# Workshop çıktısını kalabalıklaştıran genel uyarılar gizlenir.

import numpy as np
# Sayısal kalite kontrolü ve Lipinski sonuçları için kullanılır.

import pandas as pd
# CSV dosyalarını okumak, temizlemek ve kaydetmek için kullanılır.

import requests
# GitHub API ve RAW dosya bağlantılarına istek göndermek için kullanılır.

from tqdm.auto import tqdm
# SMILES standardizasyonunda ilerleme çubuğu göstermek için kullanılır.

from IPython.display import display
# DataFrame tablolarını Colab içinde okunabilir biçimde göstermek için kullanılır.

from google.colab import drive
# Sonuç dosyalarını Google Drive'a kaydetmek için kullanılır.

from rdkit import Chem
# SMILES metinlerini RDKit molekül nesnesine dönüştürmek için kullanılır.

from rdkit.Chem import Crippen
# MolLogP değerini hesaplamak için kullanılır.

from rdkit.Chem import Descriptors
# Molekül ağırlığını hesaplamak için kullanılır.

from rdkit.Chem import Lipinski
# Hidrojen bağı verici ve alıcı sayılarını hesaplamak için kullanılır.

from rdkit.Chem.MolStandardize import rdMolStandardize
# Tuzları ve küçük karşı iyonları uzaklaştırarak parent yapı üretmek için kullanılır.


tqdm.pandas()
# pandas progress_apply işlemlerinde ilerleme çubuğu etkinleştirilir.


print("RDKit sürümü:", Chem.rdBase.rdkitVersion)
# Kullanılan RDKit sürümü kontrol amacıyla yazdırılır.

print("Pandas sürümü:", pd.__version__)
# Kullanılan pandas sürümü kontrol amacıyla yazdırılır.

print("✅ CHECKPOINT 01.2: Kütüphaneler başarıyla içe aktarıldı.")
# Import hücresinin tamamlandığı bildirilir.

## Hücre 3 — Google Drive bağlantısı ve çalışma ayarları

In [ ]:
drive.mount("/content/drive", force_remount=False)
# Google Drive standart Colab dizinine bağlanır.


TARGET_ID = "CHEMBL206"
# İşlenecek ChEMBL Target ID burada değiştirilir.


GITHUB_OWNER = "MOL-FEST"
# GitHub organizasyon veya kullanıcı adı tanımlanır.


GITHUB_REPOSITORY = "MOL-FET_regression"
# GitHub repo adı tanımlanır.


GITHUB_BRANCH = "main"
# Dosyaların okunacağı GitHub branch adı tanımlanır.


TARGET_INPUT_FILENAME = (
    f"{TARGET_ID}_IC50_single_protein_format_CLEAN_parent_dedup.csv"
)
# Target ID'ye göre tercih edilen ana girdi dosya adı oluşturulur.


DRIVE_ROOT = Path(
    "/content/drive/MyDrive/MOL_FET_regression_workshop"
)
# Workshop çıktılarının kaydedileceği ortak Drive klasörü tanımlanır.


DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
# Drive çıktı klasörü yoksa oluşturulur.


MAX_LIPINSKI_VIOLATIONS = 0
# Sıfır ihlal, strict Lipinski Rule-of-Five filtresi olarak kullanılır.


REQUEST_TIMEOUT = 180
# GitHub istekleri için saniye cinsinden zaman aşımı belirlenir.


print("TARGET_ID:", TARGET_ID)
# Aktif Target ID ekrana yazdırılır.

print("GitHub repo:", f"{GITHUB_OWNER}/{GITHUB_REPOSITORY}")
# Kullanılan GitHub deposu yazdırılır.

print("Drive çıktı klasörü:", DRIVE_ROOT)
# Çıktı klasörü kullanıcıya gösterilir.

print("✅ CHECKPOINT 01.3: Drive ve çalışma ayarları hazır.")
# Ayar hücresinin tamamlandığı bildirilir.

## Hücre 4 — GitHub dosya adaylarının tanımlanması

Notebook aşağıdaki dosya konumlarını öncelik sırasıyla değerlendirir:

1. `data/<TARGET_ID>_IC50_single_protein_format_CLEAN_parent_dedup.csv`
2. `<TARGET_ID>_IC50_single_protein_format_CLEAN_parent_dedup.csv`
3. `training_data.csv`
4. `data/training_data.csv`

Ek olarak GitHub API ile repo ağacında aynı dosya adları aranır.

In [ ]:
PREFERRED_GITHUB_PATHS = [
    f"data/{TARGET_INPUT_FILENAME}",
    # Workshop veri yapısında tercih edilen data/ klasörü yolu.

    TARGET_INPUT_FILENAME,
    # Dosya repo kök dizinine yüklenmişse kullanılacak yol.

    "training_data.csv",
    # Mevcut MOL-FET regression reposundaki genel eğitim verisi için aday yol.

    "data/training_data.csv",
    # training_data.csv daha sonra data/ klasörüne taşınırsa kullanılacak yol.
]
# GitHub içinde aranacak göreli dosya yolları öncelik sırasıyla tanımlanır.


GITHUB_API_TREE_URL = (
    f"https://api.github.com/repos/{GITHUB_OWNER}/"
    f"{GITHUB_REPOSITORY}/git/trees/{GITHUB_BRANCH}?recursive=1"
)
# GitHub deposundaki bütün dosyaları listeleyen API adresi oluşturulur.


def build_raw_url(relative_path):
    """GitHub göreli dosya yolunu RAW indirme adresine dönüştürür."""

    clean_path = str(relative_path).lstrip("/")
    # Yolun başındaki olası eğik çizgi kaldırılır.

    return (
        f"https://raw.githubusercontent.com/{GITHUB_OWNER}/"
        f"{GITHUB_REPOSITORY}/{GITHUB_BRANCH}/{clean_path}"
    )
    # GitHub RAW indirme adresi döndürülür.


print("Öncelikli GitHub dosya adayları:")
# Aday dosyaların başlığı yazdırılır.

for candidate_path in PREFERRED_GITHUB_PATHS:
    # Her aday yol sırayla gösterilir.

    print(" -", candidate_path)
    # Aday repo yolu ekrana yazdırılır.


print("✅ CHECKPOINT 01.4: GitHub dosya adayları tanımlandı.")
# Aday tanımlama hücresinin tamamlandığı bildirilir.

## Hücre 5 — GitHub repo ağacını okuyan yardımcı fonksiyonlar

In [ ]:
def get_repository_file_paths():
    """GitHub API üzerinden repodaki bütün dosya yollarını döndürür."""

    print("GitHub repo ağacı kontrol ediliyor...")
    # Kullanıcıya API kontrolünün başladığı bildirilir.

    response = requests.get(
        GITHUB_API_TREE_URL,
        timeout=REQUEST_TIMEOUT,
        headers={
            "Accept": "application/vnd.github+json",
        },
    )
    # GitHub API üzerinden recursive repo ağacı istenir.

    response.raise_for_status()
    # API 404, rate-limit veya bağlantı hatasında açıklayıcı exception oluşturulur.

    payload = response.json()
    # JSON yanıtı Python sözlüğüne dönüştürülür.

    tree_entries = payload.get("tree", [])
    # Repo ağacındaki girişler alınır.

    file_paths = [
        entry.get("path")
        for entry in tree_entries
        if entry.get("type") == "blob" and entry.get("path")
    ]
    # Yalnızca gerçek dosya girişlerinin göreli yolları tutulur.

    if not file_paths:
        # API yanıtında dosya bulunamazsa kontrol edilir.

        raise RuntimeError(
            "GitHub API repo ağacında hiçbir dosya döndürmedi."
        )
        # Boş repo ağacıyla devam edilmez.

    return file_paths
    # Repo içindeki bütün dosya yolları döndürülür.


def resolve_github_csv_path():
    """Repo içinde kullanılabilir regression CSV yolunu otomatik bulur."""

    try:
        # Öncelikle GitHub API ile repo ağacı okunmaya çalışılır.

        repository_files = get_repository_file_paths()
        # Repo içindeki dosya yolları alınır.

        repository_file_set = set(repository_files)
        # Hızlı tam yol kontrolü için dosya yolları kümeye dönüştürülür.

        for preferred_path in PREFERRED_GITHUB_PATHS:
            # Öncelikli adaylar sırayla kontrol edilir.

            if preferred_path in repository_file_set:
                # Aday yol gerçekten repoda bulunuyorsa seçilir.

                print("GitHub API ile bulunan dosya:", preferred_path)
                # Bulunan yol ekrana yazdırılır.

                return preferred_path
                # İlk uygun aday yol döndürülür.

        preferred_names = {
            TARGET_INPUT_FILENAME,
            "training_data.csv",
        }
        # Repo içinde alt klasör değişmiş olsa bile aranacak dosya adları tanımlanır.

        recursive_matches = [
            path
            for path in repository_files
            if Path(path).name in preferred_names
        ]
        # Dosya adı eşleşen bütün recursive sonuçlar seçilir.

        if recursive_matches:
            # En az bir recursive eşleşme varsa kontrol edilir.

            recursive_matches = sorted(
                recursive_matches,
                key=lambda path: (
                    Path(path).name != TARGET_INPUT_FILENAME,
                    len(Path(path).parts),
                    path,
                ),
            )
            # Target'a özel dosya ve daha kısa yollar öncelikli olacak biçimde sıralanır.

            selected_path = recursive_matches[0]
            # En uygun eşleşme seçilir.

            print("Recursive arama ile bulunan dosya:", selected_path)
            # Bulunan yol ekrana yazdırılır.

            return selected_path
            # Seçilen GitHub yolu döndürülür.

        csv_files = sorted(
            path
            for path in repository_files
            if str(path).lower().endswith(".csv")
        )
        # Tanı koymaya yardımcı olmak için repodaki bütün CSV yolları listelenir.

        raise FileNotFoundError(
            "Beklenen regression CSV GitHub reposunda bulunamadı.\n"
            "Repoda bulunan CSV dosyaları:\n- "
            + "\n- ".join(csv_files[:50])
        )
        # Hiç uygun dosya yoksa mevcut CSV listesiyle açıklayıcı hata üretilir.

    except requests.RequestException as api_error:
        # GitHub API erişilemezse doğrudan RAW adaylarını denemeye geçilir.

        print("GitHub API kontrolü başarısız:", api_error)
        # API hatası bilgi amaçlı gösterilir.

        print("RAW bağlantı adayları sırayla denenecek.")
        # Fallback yöntemine geçildiği bildirilir.

        return None
        # API sonucu olmadan fallback aşamasına geçilir.


print("✅ CHECKPOINT 01.5: GitHub dosya çözümleme fonksiyonları hazır.")
# Yardımcı fonksiyonların tanımlandığı bildirilir.

## Hücre 6 — CSV indirme ve sütun doğrulama fonksiyonları

In [ ]:
def dataframe_has_required_columns(dataframe):
    """DataFrame'in temizlik için gerekli sütunları içerip içermediğini kontrol eder."""

    required_columns = {
        "parent_smiles",
        "pStandard",
    }
    # Temizlik pipeline'ı için zorunlu sütunlar tanımlanır.

    return required_columns.issubset(dataframe.columns)
    # Her iki zorunlu sütun mevcutsa True döndürülür.


def download_and_validate_csv(relative_path):
    """Bir GitHub yolundaki CSV'yi indirir ve yapısını doğrular."""

    raw_url = build_raw_url(relative_path)
    # Göreli GitHub yolu RAW URL adresine dönüştürülür.

    print("Denenen GitHub RAW URL:", raw_url)
    # Test edilen tam adres ekrana yazdırılır.

    response = requests.get(
        raw_url,
        timeout=REQUEST_TIMEOUT,
    )
    # GitHub RAW adresinden dosya istenir.

    if response.status_code == 404:
        # Dosya bu yolda bulunmuyorsa kontrol edilir.

        print("  → 404: Bu konumda dosya yok.")
        # 404 sonucu açıklayıcı biçimde yazdırılır.

        return None, raw_url
        # Pipeline durdurulmadan bir sonraki adaya geçilmesi sağlanır.

    response.raise_for_status()
    # 404 dışındaki HTTP hatalarında exception oluşturulur.

    if len(response.content) == 0:
        # İndirilen dosyanın boş olup olmadığı kontrol edilir.

        print("  → Dosya boş; sonraki aday denenecek.")
        # Boş dosya bilgisi yazdırılır.

        return None, raw_url
        # Boş aday kullanılmaz.

    try:
        # CSV ayrıştırma hatasını yakalamak için try bloğu başlatılır.

        dataframe = pd.read_csv(
            BytesIO(response.content),
            low_memory=False,
        )
        # GitHub içeriği bellekte pandas DataFrame'e dönüştürülür.

    except Exception as csv_error:
        # İndirilen içerik CSV olarak okunamazsa hata yakalanır.

        print("  → CSV okuma hatası:", csv_error)
        # Hata bilgisi ekrana yazdırılır.

        return None, raw_url
        # Geçersiz dosya yerine sonraki aday denenir.

    if dataframe.empty:
        # DataFrame boşsa kontrol edilir.

        print("  → CSV boş; sonraki aday denenecek.")
        # Boş CSV bilgisi yazdırılır.

        return None, raw_url
        # Boş veri kullanılmaz.

    if not dataframe_has_required_columns(dataframe):
        # Zorunlu sütunların bulunup bulunmadığı kontrol edilir.

        print(
            "  → Gerekli sütunlar yok. Bulunan sütunlar:",
            list(dataframe.columns),
        )
        # Uygun olmayan CSV'nin sütunları tanı için gösterilir.

        return None, raw_url
        # Modelleme yapısına uymayan CSV atlanır.

    print("  → Uygun regression CSV bulundu.")
    # Başarılı aday bilgisi yazdırılır.

    return dataframe, raw_url
    # Doğrulanmış DataFrame ve kullanılan URL döndürülür.


print("✅ CHECKPOINT 01.6: CSV indirme ve doğrulama fonksiyonları hazır.")
# Fonksiyon hücresinin tamamlandığı bildirilir.

## Hücre 7 — GitHub'daki uygun regression CSV'nin otomatik seçilmesi

In [ ]:
resolved_path = resolve_github_csv_path()
# GitHub API ile mümkün olan en uygun CSV yolu bulunur.


paths_to_try = []
# Denenecek GitHub yollarını tutacak boş liste oluşturulur.


if resolved_path is not None:
    # API bir yol bulduysa kontrol edilir.

    paths_to_try.append(resolved_path)
    # API tarafından bulunan yol ilk sıraya eklenir.


for preferred_path in PREFERRED_GITHUB_PATHS:
    # Önceden tanımlanan bütün aday yollar sırayla dolaşılır.

    if preferred_path not in paths_to_try:
        # Aynı yol daha önce listeye eklenmediyse kontrol edilir.

        paths_to_try.append(preferred_path)
        # Aday yol deneme listesine eklenir.


df_raw = None
# Başlangıçta henüz veri okunmadığı belirtilir.


USED_GITHUB_URL = None
# Kullanılan GitHub RAW URL daha sonra raporlamak için boş tanımlanır.


for candidate_path in paths_to_try:
    # Bütün aday GitHub yolları sırayla denenir.

    candidate_df, candidate_url = download_and_validate_csv(
        candidate_path
    )
    # Aday dosya indirilir ve sütun yapısı kontrol edilir.

    if candidate_df is not None:
        # Geçerli DataFrame bulunduysa kontrol edilir.

        df_raw = candidate_df
        # Geçerli veri ana DataFrame olarak atanır.

        USED_GITHUB_URL = candidate_url
        # Başarılı RAW URL saklanır.

        break
        # İlk uygun veri bulunduğunda döngü sonlandırılır.


if df_raw is None:
    # Hiçbir adaydan geçerli veri okunamadıysa kontrol edilir.

    raise FileNotFoundError(
        "GitHub reposunda parent_smiles ve pStandard sütunlarını "
        "içeren uygun bir CSV bulunamadı. "
        "CSV dosyasını repo köküne training_data.csv adıyla veya "
        f"data/{TARGET_INPUT_FILENAME} adıyla yükleyiniz."
    )
    # Kullanıcıya dosyanın nereye yüklenmesi gerektiğini açıklayan hata oluşturulur.


print("Kullanılan GitHub RAW URL:", USED_GITHUB_URL)
# Başarılı kaynak URL ekrana yazdırılır.

print("Ham veri boyutu:", df_raw.shape)
# Ham veri setinin satır ve sütun sayısı yazdırılır.

display(df_raw.head(10))
# İlk 10 kayıt workshop kontrolü için gösterilir.

print("✅ CHECKPOINT 01.7: GitHub regression CSV başarıyla okundu.")
# GitHub okuma adımının tamamlandığı bildirilir.

## Hücre 8 — Girdi sütunlarının kalite kontrolü

In [ ]:
required_columns = {
    "parent_smiles",
    "pStandard",
}
# Pipeline için zorunlu minimum sütunlar tanımlanır.


missing_columns = required_columns.difference(df_raw.columns)
# Eksik zorunlu sütunlar belirlenir.


if missing_columns:
    # Eksik sütun olup olmadığı kontrol edilir.

    raise KeyError(
        f"GitHub CSV içinde eksik sütunlar: {sorted(missing_columns)}"
    )
    # Eksik zorunlu sütun varsa pipeline durdurulur.


df_raw = df_raw.copy()
# GitHub'dan okunan orijinal DataFrame korunarak kopya oluşturulur.


df_raw["pStandard"] = pd.to_numeric(
    df_raw["pStandard"],
    errors="coerce",
)
# Target sütunu güvenli biçimde sayısal değere dönüştürülür.


df_raw["parent_smiles"] = (
    df_raw["parent_smiles"]
    .astype("string")
    .str.strip()
)
# SMILES sütunu metne dönüştürülür ve baştaki/sondaki boşluklar temizlenir.


missing_target_count = int(df_raw["pStandard"].isna().sum())
# Eksik veya sayıya dönüşmeyen target sayısı hesaplanır.


missing_smiles_count = int(
    df_raw["parent_smiles"].fillna("").eq("").sum()
)
# Eksik veya boş SMILES sayısı hesaplanır.


df_input = df_raw.dropna(
    subset=[
        "parent_smiles",
        "pStandard",
    ]
).copy()
# Zorunlu SMILES veya target bilgisi eksik satırlar çalışma setinden çıkarılır.


df_input = df_input.loc[
    df_input["parent_smiles"].astype(str).str.len() > 0
].copy()
# Boş metin olarak kalan SMILES satırları ayrıca çıkarılır.


if df_input.empty:
    # Zorunlu alan temizliği sonrasında veri kalıp kalmadığı kontrol edilir.

    raise ValueError(
        "SMILES ve pStandard kalite kontrolünden sonra veri kalmadı."
    )
    # Boş veri setiyle devam edilmez.


print("Eksik/geçersiz pStandard:", missing_target_count)
# Eksik target sayısı yazdırılır.

print("Eksik/boş parent_smiles:", missing_smiles_count)
# Eksik SMILES sayısı yazdırılır.

print("Kalite kontrolü sonrası satır:", len(df_input))
# Temel kalite kontrolünden geçen satır sayısı yazdırılır.

print("pStandard aralığı:", df_input["pStandard"].min(), "—", df_input["pStandard"].max())
# Target değer aralığı kontrol amacıyla gösterilir.

print("✅ CHECKPOINT 01.8: Girdi sütunları ve target değerleri doğrulandı.")
# Temel veri kalite kontrolünün tamamlandığı bildirilir.

## Hücre 9 — Tuz temizliği ve canonical parent SMILES fonksiyonu

In [ ]:
def standardize_to_parent(smiles):
    """SMILES'ı doğrular, tuzları çıkarır ve canonical parent SMILES üretir."""

    if pd.isna(smiles):
        # Eksik SMILES kontrol edilir.

        return None
        # Eksik değer geçersiz kabul edilir.

    text = str(smiles).strip()
    # SMILES temiz bir metne dönüştürülür.

    if not text:
        # Boş metin kontrol edilir.

        return None
        # Boş SMILES geçersiz kabul edilir.

    try:
        # RDKit ayrıştırma ve standardizasyon hatalarını yakalamak için try başlatılır.

        molecule = Chem.MolFromSmiles(text)
        # SMILES RDKit molekül nesnesine dönüştürülür.

        if molecule is None:
            # RDKit molekül oluşturamazsa kontrol edilir.

            return None
            # Ayrıştırılamayan SMILES geçersiz kabul edilir.

        parent = rdMolStandardize.FragmentParent(molecule)
        # Tuz ve küçük karşı iyonlar uzaklaştırılarak ana moleküler fragman seçilir.

        Chem.SanitizeMol(parent)
        # Parent molekülün valans ve aromatiklik gibi kimyasal kuralları doğrulanır.

        canonical_smiles = Chem.MolToSmiles(
            parent,
            canonical=True,
            isomericSmiles=True,
        )
        # Parent yapı canonical ve stereokimyayı koruyan SMILES'a dönüştürülür.

        return canonical_smiles
        # Standardize edilmiş parent SMILES döndürülür.

    except Exception:
        # Beklenmeyen RDKit hataları yakalanır.

        return None
        # Hatalı yapı geçersiz olarak işaretlenir.


print("✅ CHECKPOINT 01.9: Parent SMILES standardizasyon fonksiyonu hazır.")
# Fonksiyon tanımının tamamlandığı bildirilir.

## Hücre 10 — Tuz ve SMILES standardizasyonunun uygulanması

In [ ]:
df_work = df_input.copy()
# Kalite kontrolünden geçen veri korunarak çalışma kopyası oluşturulur.


df_work["had_multiple_fragments"] = (
    df_work["parent_smiles"]
    .astype(str)
    .str.contains(".", regex=False)
)
# Nokta karakteri içeren olası tuz veya çoklu fragman kayıtları işaretlenir.


df_work["canonical_parent_smiles"] = (
    df_work["parent_smiles"]
    .progress_apply(standardize_to_parent)
)
# Her girdi SMILES için tuzlardan arındırılmış canonical parent SMILES üretilir.


df_work["is_valid_smiles"] = (
    df_work["canonical_parent_smiles"].notna()
)
# Geçerli parent SMILES üretilebilen satırlar işaretlenir.


invalid_smiles = df_work.loc[
    ~df_work["is_valid_smiles"]
].copy()
# RDKit tarafından standardize edilemeyen kayıtlar ayrı rapora alınır.


valid_df = df_work.loc[
    df_work["is_valid_smiles"]
].copy()
# Yalnızca geçerli canonical parent yapılara sahip satırlarla devam edilir.


if valid_df.empty:
    # Standardizasyon sonrasında geçerli yapı kalıp kalmadığı kontrol edilir.

    raise ValueError(
        "SMILES standardizasyonundan sonra geçerli molekül kalmadı."
    )
    # Geçerli molekül yoksa pipeline durdurulur.


print("Başlangıç satırı:", len(df_input))
# Standardizasyon öncesi satır sayısı yazdırılır.

print(
    "Çoklu fragman/tuz göstergeli satır:",
    int(df_work["had_multiple_fragments"].sum()),
)
# Noktalı SMILES sayısı yazdırılır.

print("Geçerli standardize SMILES:", len(valid_df))
# Geçerli yapı sayısı yazdırılır.

print("Geçersiz SMILES:", len(invalid_smiles))
# Geçersiz yapı sayısı yazdırılır.

display(
    valid_df[
        [
            "parent_smiles",
            "canonical_parent_smiles",
            "pStandard",
        ]
    ].head(10)
)
# Ham ve canonical SMILES karşılaştırması gösterilir.

print("✅ CHECKPOINT 01.10: Tuz ve SMILES standardizasyonu tamamlandı.")
# Standardizasyon adımının tamamlandığı bildirilir.

## Hücre 11 — Duplicate birleştirme yardımcı fonksiyonu

In [ ]:
def join_unique(values):
    """Bir gruptaki benzersiz metin değerlerini noktalı virgülle birleştirir."""

    clean_values = sorted(
        {
            str(value)
            for value in values
            if pd.notna(value)
        }
    )
    # Eksik olmayan benzersiz değerler metne çevrilir ve sıralanır.

    return ";".join(clean_values)
    # Değerler izlenebilir biçimde tek bir metinde birleştirilir.


print("✅ CHECKPOINT 01.11: Duplicate birleştirme yardımcı fonksiyonu hazır.")
# Yardımcı fonksiyonun tanımlandığı bildirilir.

## Hücre 12 — Canonical duplicate kayıtların birleştirilmesi

In [ ]:
aggregation = {
    "pStandard": ("pStandard", "median"),
    # Final target değeri olarak medyan pStandard kullanılır.

    "pStandard_mean": ("pStandard", "mean"),
    # Aynı yapıya ait ölçümlerin ortalaması saklanır.

    "pStandard_std": ("pStandard", "std"),
    # Aynı yapıya ait ölçümlerin standart sapması saklanır.

    "pStandard_min": ("pStandard", "min"),
    # En düşük pStandard değeri saklanır.

    "pStandard_max": ("pStandard", "max"),
    # En yüksek pStandard değeri saklanır.

    "source_rows": ("pStandard", "size"),
    # Canonical yapıya birleşen kaynak satır sayısı saklanır.

    "had_multiple_fragments": ("had_multiple_fragments", "max"),
    # Kaynak satırlardan herhangi birinde tuz göstergesi olup olmadığı saklanır.
}
# GroupBy aggregation işlemleri tanımlanır.


if "parent_chembl_id" in valid_df.columns:
    # Parent ChEMBL ID sütunu mevcutsa kontrol edilir.

    aggregation["parent_chembl_id"] = (
        "parent_chembl_id",
        join_unique,
    )
    # Birleşen ChEMBL ID değerleri kaybedilmeden saklanır.


if "n_measurements" in valid_df.columns:
    # Kaynak veri ölçüm sayısını içeriyorsa kontrol edilir.

    aggregation["source_n_measurements"] = (
        "n_measurements",
        "sum",
    )
    # Kaynak ölçüm sayıları toplanarak bilgi amaçlı saklanır.


if "n_assays" in valid_df.columns:
    # Kaynak veri assay sayısını içeriyorsa kontrol edilir.

    aggregation["source_n_assays"] = (
        "n_assays",
        "sum",
    )
    # Kaynak assay sayıları bilgi amaçlı toplanır.


if "n_molecule_ids" in valid_df.columns:
    # Kaynak veri molekül ID sayısını içeriyorsa kontrol edilir.

    aggregation["source_n_molecule_ids"] = (
        "n_molecule_ids",
        "sum",
    )
    # Kaynak molekül ID sayıları bilgi amaçlı toplanır.


df_dedup = (
    valid_df
    .groupby(
        "canonical_parent_smiles",
        as_index=False,
    )
    .agg(**aggregation)
    .rename(
        columns={
            "canonical_parent_smiles": "parent_smiles",
        }
    )
)
# Aynı canonical parent SMILES'a dönüşen kayıtlar tek satırda birleştirilir.


df_dedup["pStandard_std"] = (
    df_dedup["pStandard_std"].fillna(0.0)
)
# Tek ölçümlü moleküllerde oluşan NaN standart sapma sıfırla doldurulur.


df_dedup["pStandard_range"] = (
    df_dedup["pStandard_max"]
    - df_dedup["pStandard_min"]
)
# Aynı yapıya ait en yüksek ve en düşük target farkı hesaplanır.


df_dedup["activity_conflict_flag"] = (
    df_dedup["pStandard_range"] > 1.0
)
# Bir log birimden büyük ölçüm farkı gösteren yapılar işaretlenir.


if not df_dedup["parent_smiles"].is_unique:
    # Dedup sonrasında tekrar kalıp kalmadığı kontrol edilir.

    raise AssertionError(
        "Canonical parent SMILES tekrarları tamamen birleştirilemedi."
    )
    # Tekrar kaldıysa pipeline durdurulur.


print("Standardizasyon öncesi geçerli satır:", len(valid_df))
# Duplicate öncesi kayıt sayısı yazdırılır.

print("Duplicate sonrası benzersiz molekül:", len(df_dedup))
# Birleştirme sonrası benzersiz yapı sayısı yazdırılır.

print(
    "Aktivite çatışması işaretli molekül:",
    int(df_dedup["activity_conflict_flag"].sum()),
)
# Yüksek deneysel saçılım gösteren yapı sayısı yazdırılır.

print("✅ CHECKPOINT 01.12: Canonical duplicate kayıtlar birleştirildi.")
# Duplicate birleştirme adımının tamamlandığı bildirilir.

## Hücre 13 — Lipinski Rule of Five fonksiyonu

In [ ]:
def calculate_lipinski(smiles):
    """Bir parent SMILES için Lipinski Rule-of-Five değerlerini hesaplar."""

    molecule = Chem.MolFromSmiles(smiles)
    # Parent SMILES RDKit molekül nesnesine dönüştürülür.

    if molecule is None:
        # Molekül oluşturulamazsa kontrol edilir.

        return {
            "MolWt": np.nan,
            "MolLogP": np.nan,
            "HBD": np.nan,
            "HBA": np.nan,
            "Lipinski_violations": np.nan,
            "passes_lipinski": False,
        }
        # Geçersiz molekül için boş descriptor sonuçları döndürülür.

    mol_wt = Descriptors.MolWt(molecule)
    # Molekül ağırlığı Dalton cinsinden hesaplanır.

    mol_logp = Crippen.MolLogP(molecule)
    # Hesaplanan oktanol/su LogP değeri hesaplanır.

    hbd = Lipinski.NumHDonors(molecule)
    # Hidrojen bağı verici grup sayısı hesaplanır.

    hba = Lipinski.NumHAcceptors(molecule)
    # Hidrojen bağı alıcı grup sayısı hesaplanır.

    violations = (
        int(mol_wt > 500)
        + int(mol_logp > 5)
        + int(hbd > 5)
        + int(hba > 10)
    )
    # Dört klasik Lipinski sınırından kaçının ihlal edildiği hesaplanır.

    passes_lipinski = (
        violations <= MAX_LIPINSKI_VIOLATIONS
    )
    # İhlal sayısının izin verilen maksimum değeri aşıp aşmadığı kontrol edilir.

    return {
        "MolWt": mol_wt,
        "MolLogP": mol_logp,
        "HBD": hbd,
        "HBA": hba,
        "Lipinski_violations": violations,
        "passes_lipinski": passes_lipinski,
    }
    # Lipinski descriptorları ve filtre sonucu döndürülür.


print("✅ CHECKPOINT 01.13: Lipinski hesaplama fonksiyonu hazır.")
# Lipinski fonksiyonunun tanımlandığı bildirilir.

## Hücre 14 — Lipinski descriptorlarının hesaplanması ve filtrenin uygulanması

In [ ]:
lipinski_df = pd.DataFrame(
    df_dedup["parent_smiles"]
    .progress_apply(calculate_lipinski)
    .tolist()
)
# Her benzersiz parent yapı için Lipinski özellikleri hesaplanır.


df_all = pd.concat(
    [
        df_dedup.reset_index(drop=True),
        lipinski_df.reset_index(drop=True),
    ],
    axis=1,
)
# Duplicate tablosu ile Lipinski descriptorları yatay olarak birleştirilir.


df_filtered = df_all.loc[
    df_all["passes_lipinski"]
].copy()
# Belirlenen maksimum ihlal sayısını aşmayan moleküller seçilir.


df_rejected = df_all.loc[
    ~df_all["passes_lipinski"]
].copy()
# Lipinski filtresinden geçmeyen moleküller ayrı rapora alınır.


if df_filtered.empty:
    # Lipinski filtresi sonrasında molekül kalıp kalmadığı kontrol edilir.

    raise ValueError(
        "Lipinski filtresinden sonra kullanılabilir molekül kalmadı."
    )
    # Hiç molekül kalmadıysa pipeline durdurulur.


print("Lipinski öncesi benzersiz molekül:", len(df_all))
# Filtre öncesi molekül sayısı yazdırılır.

print("Lipinski'den geçen:", len(df_filtered))
# Filtreyi geçen molekül sayısı yazdırılır.

print("Lipinski'den kalan:", len(df_rejected))
# Filtreyi geçemeyen molekül sayısı yazdırılır.


display(
    df_all["Lipinski_violations"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("ihlal_sayısı")
    .to_frame("molekül_sayısı")
)
# Lipinski ihlal sayısı dağılımı tablo olarak gösterilir.


display(df_filtered.head(10))
# Filtrelenmiş veri setinin ilk 10 satırı gösterilir.


print("✅ CHECKPOINT 01.14: Lipinski filtresi tamamlandı.")
# Lipinski adımının tamamlandığı bildirilir.

## Hücre 15 — Temizlik raporunun hazırlanması

In [ ]:
cleaning_report = pd.DataFrame(
    {
        "metric": [
            "github_source_url",
            "raw_rows",
            "rows_after_required_field_check",
            "invalid_smiles_rows",
            "valid_standardized_rows",
            "unique_parent_rows_after_dedup",
            "activity_conflict_rows",
            "lipinski_pass_rows",
            "lipinski_rejected_rows",
            "max_lipinski_violations_allowed",
        ],
        "value": [
            USED_GITHUB_URL,
            len(df_raw),
            len(df_input),
            len(invalid_smiles),
            len(valid_df),
            len(df_dedup),
            int(df_dedup["activity_conflict_flag"].sum()),
            len(df_filtered),
            len(df_rejected),
            MAX_LIPINSKI_VIOLATIONS,
        ],
    }
)
# Bütün veri temizleme aşamalarını özetleyen rapor tablosu oluşturulur.


display(cleaning_report)
# Temizlik raporu notebook içinde gösterilir.


print("✅ CHECKPOINT 01.15: Temizlik kalite raporu hazırlandı.")
# Kalite raporu adımının tamamlandığı bildirilir.

## Hücre 16 — Bütün çıktıların Google Drive'a kaydedilmesi

In [ ]:
filtered_filename = (
    f"{TARGET_ID}_IC50_parent_dedup_Lipinski_filtered.csv"
)
# Sonraki Mordred notebookunun GitHub'dan okuyacağı final CSV adı oluşturulur.


all_filename = (
    f"{TARGET_ID}_IC50_parent_dedup_with_Lipinski.csv"
)
# Bütün standardize molekülleri içeren çıktı adı oluşturulur.


invalid_filename = (
    f"{TARGET_ID}_invalid_smiles.csv"
)
# Geçersiz SMILES rapor dosya adı oluşturulur.


rejected_filename = (
    f"{TARGET_ID}_Lipinski_rejected.csv"
)
# Lipinski filtresinden geçmeyen moleküllerin dosya adı oluşturulur.


report_filename = (
    f"{TARGET_ID}_cleaning_report.csv"
)
# Temizlik özet raporu dosya adı oluşturulur.


filtered_path = DRIVE_ROOT / filtered_filename
# Final filtreli CSV için Google Drive yolu oluşturulur.


all_path = DRIVE_ROOT / all_filename
# Tüm standardize moleküller için Drive yolu oluşturulur.


invalid_path = DRIVE_ROOT / invalid_filename
# Geçersiz SMILES raporu için Drive yolu oluşturulur.


rejected_path = DRIVE_ROOT / rejected_filename
# Lipinski reddedilenler için Drive yolu oluşturulur.


report_path = DRIVE_ROOT / report_filename
# Temizlik raporu için Drive yolu oluşturulur.


df_filtered.to_csv(
    filtered_path,
    index=False,
)
# Final filtreli regression CSV Google Drive'a kaydedilir.


df_all.to_csv(
    all_path,
    index=False,
)
# Lipinski hesaplanmış bütün kayıtlar Google Drive'a kaydedilir.


invalid_smiles.to_csv(
    invalid_path,
    index=False,
)
# Geçersiz SMILES kayıtları Google Drive'a kaydedilir.


df_rejected.to_csv(
    rejected_path,
    index=False,
)
# Lipinski filtresinden geçmeyen kayıtlar Google Drive'a kaydedilir.


cleaning_report.to_csv(
    report_path,
    index=False,
)
# Temizlik kalite raporu Google Drive'a kaydedilir.


OUTPUT_PATHS = [
    filtered_path,
    all_path,
    invalid_path,
    rejected_path,
    report_path,
]
# Kontrol edilecek bütün çıktı yolları listelenir.


for output_path in OUTPUT_PATHS:
    # Üretilen her dosya sırayla kontrol edilir.

    if not output_path.exists():
        # Dosyanın gerçekten oluşup oluşmadığı kontrol edilir.

        raise IOError(
            f"Google Drive çıktısı oluşturulamadı: {output_path}"
        )
        # Eksik çıktı varsa pipeline durdurulur.

    if output_path.stat().st_size == 0:
        # Oluşturulan dosyanın boş olup olmadığı kontrol edilir.

        raise IOError(
            f"Google Drive çıktısı boş oluşturuldu: {output_path}"
        )
        # Boş çıktı kabul edilmez.

    print(
        "[Google Drive kaydı]",
        output_path,
        f"({output_path.stat().st_size:,} byte)",
    )
    # Başarılı çıktı yolu ve dosya boyutu gösterilir.


print()
# Okunabilirlik için boş satır yazdırılır.

print(
    "GitHub'a yüklenmesi gereken ana çıktı:",
    filtered_filename,
)
# Bir sonraki notebookun okuyacağı final dosya adı bildirilir.

print(
    "Önerilen GitHub konumu:",
    f"data/{filtered_filename}",
)
# Final CSV'nin GitHub'daki önerilen yolu gösterilir.

print(
    "✅ CHECKPOINT 01.16: Notebook tamamlandı; "
    "bütün çıktılar Google Drive'a kaydedildi."
)
# Notebookun başarıyla tamamlandığı bildirilir.

## Son kontrol

Notebook başarıyla tamamlandığında Google Drive içinde şu dosyalar oluşur:

```text
CHEMBL206_IC50_parent_dedup_Lipinski_filtered.csv
CHEMBL206_IC50_parent_dedup_with_Lipinski.csv
CHEMBL206_invalid_smiles.csv
CHEMBL206_Lipinski_rejected.csv
CHEMBL206_cleaning_report.csv
```

Bir sonraki Mordred notebooku için GitHub'a yüklenmesi gereken ana dosya:

```text
data/CHEMBL206_IC50_parent_dedup_Lipinski_filtered.csv
```